# 02 The Data Model

This notebook audits the major record types exposed by TorchLens 2.0. An `Op` is one captured tensor-producing operation; a `Layer` is the stable graph position that can aggregate ops; modules, params, buffers, and grad functions provide the rest of the execution anatomy.

The example model includes parameters, a buffer, and autograd metadata. We capture with gradient saving enabled, then explicitly log backward metadata.

In [ ]:
from pathlib import Path
import sys

REPO_ROOT = Path.cwd()
if REPO_ROOT.name == "audit":
    REPO_ROOT = REPO_ROOT.parents[1]
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(f"repo root: {REPO_ROOT}")

In [ ]:
import torch
from torch import nn
import torchlens as tl

torch.manual_seed(13)


class AnatomyNet(nn.Module):
    """Tiny module that exposes params, a buffer, and backward metadata."""

    def __init__(self) -> None:
        """Initialize layers and a buffer."""
        super().__init__()
        self.proj = nn.Linear(3, 4)
        self.norm = nn.LayerNorm(4)
        self.head = nn.Linear(4, 1)
        self.register_buffer("bias_shift", torch.tensor([0.1]))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Run one scalar-producing forward pass."""
        h = self.norm(torch.relu(self.proj(x)))
        return self.head(h) + self.bias_shift


model = AnatomyNet().eval()
x = torch.randn(2, 3, requires_grad=True)
trace = tl.trace(model, x, gradients_to_save="all", save_code_context=True)
loss = trace[trace.output_layers[0]].out.sum()
trace.log_backward(loss)
print(trace.summary(fields=["name", "shape", "params"]))

An `Op` is the most concrete record: one operation, one output tensor, and graph links to parents and children.

In [ ]:
op = trace[1]
print(f"record type: {type(op).__name__}")
print(f"layer label: {op.layer_label}")
print(f"function: {op.func_name}")
print(f"shape/dtype: {op.shape} / {op.dtype}")
print(f"parents -> children: {op.parents} -> {op.children}")
print(f"module call: {op.module}")
print(f"trace back-pointer is trace: {op.trace is trace}")

A `Layer` is the user-facing graph position. In this single-pass model each layer has one op, but the layer object is the stable place to inspect shape, module, params, and saved output.

In [ ]:
layer = trace.layers[op.layer_label]
print(f"record type: {type(layer).__name__}")
print(f"layer label: {layer.layer_label}")
print(f"ops in this layer: {list(layer.ops)}")
print(f"saved output shape: {tuple(layer.out.shape)}")
print(f"module call label: {layer.module}")
print(f"param addresses: {[param.address for param in layer.params]}")
print(f"trace back-pointer is trace: {layer.trace is trace}")

A `Module` aggregates all calls for one PyTorch module address. A `ModuleCall` is one concrete forward invocation of that module in this trace.

In [ ]:
for address in ["proj", "norm", "head"]:
    module = trace.modules[address]
    print(
        f"Module {module.address}: class={module.class_name} inputs={module.input_layers} outputs={module.output_layers}"
    )
    for position in range(len(module.calls)):
        call = module.calls[position]
        print(f"  call {call.call_label}: ops={list(call.ops)} outputs={call.output_layers}")

`Param` and `Buffer` records connect model state to graph use. Parameters expose trainability and shape; buffers expose non-parameter tensors such as running statistics or registered constants.

In [ ]:
print("Parameters:")
for param in trace.params:
    print(
        f"  {param.address:18s} shape={param.shape} trainable={param.trainable} "
        f"module={param.module_address} memory={param.param_memory}"
    )

print("\nBuffers:")
for buffer in trace.buffers:
    print(
        f"  {buffer.address:18s} shape={buffer.shape} module={buffer.module} "
        f"memory={buffer.activation_memory}"
    )

After `log_backward`, TorchLens also exposes `GradFn` and `GradFnCall` records. These describe the PyTorch autograd graph and connect backward nodes back to forward ops when possible.

In [ ]:
print(f"grad fns: {len(trace.grad_fns)}")
print(f"grad fn calls: {len(trace.grad_fn_calls)}")
for grad_fn in list(trace.grad_fns)[:5]:
    print(
        f"  {grad_fn.label:18s} type={grad_fn.type:18s} "
        f"op_label={grad_fn.op_label} calls={grad_fn.num_calls}"
    )
for grad_call in list(trace.grad_fn_calls)[:3]:
    print(f"  call {grad_call.call_label}: grad_fn={grad_call.label} saved={grad_call.is_saved}")

Cross-references are the core of the data model: records point back to the parent `Trace`, and label fields let you resolve related records with `trace[...]`.

In [ ]:
output_label = trace.modules["head"].output_layers[0]
output_layer = trace.layers[output_label]
print(f"head output layer label: {output_label}")
print(f"resolved layer type: {type(output_layer).__name__}")
print(f"trace[label] returns: {type(trace[output_label]).__name__}")
print(f"resolved layer trace back-pointer: {output_layer.trace is trace}")
print(
    f"parents resolved through trace: {[trace[parent].layer_label for parent in output_layer.parents]}"
)

## Containers, Recurrence, Distances, and ModuleCall Arguments
A tuple/dict output records the executable container metadata currently exposed by this build. The table below separates expected intent from actual fields so duplicated output labels, missing nested container paths, and missing distance values are documented as current-build gaps rather than implied successes. Reusing the same submodule in a loop still produces one rolled layer with multiple ops and `equivalent_ops`. Module calls expose argument summaries, while saved argument templates are not retained here.


In [ ]:
class ArgBlock(nn.Module):
    """Module with positional and keyword forward arguments."""

    def forward(self, x: torch.Tensor, scale: float, *, bias: torch.Tensor) -> torch.Tensor:
        """Scale and shift the input."""
        return x * scale + bias


class ContainerRecurrentNet(nn.Module):
    """Model for container outputs and repeated submodule calls."""

    def __init__(self) -> None:
        """Initialize shared and argument-aware modules."""
        super().__init__()
        self.shared = nn.Linear(3, 3)
        self.arg_block = ArgBlock()

    def forward(self, x: torch.Tensor) -> tuple[torch.Tensor, dict[str, torch.Tensor]]:
        """Run the shared layer repeatedly and return a nested container."""
        hidden = x
        first = None
        for _ in range(3):
            hidden = torch.relu(self.shared(hidden))
            if first is None:
                first = hidden
        shifted = self.arg_block(hidden, 0.5, bias=torch.ones_like(hidden) * 0.1)
        return shifted, {"first_hidden": first, "last_hidden": hidden}


def field(obj: object, name: str, default: object = None) -> object:
    """Return an optional record field without triggering notebook failure."""
    return getattr(obj, name, default)


container_trace = tl.trace(
    ContainerRecurrentNet().eval(),
    torch.randn(2, 3),
    layers_to_save="all",
    save_arg_values=True,
    compute_input_output_distances=True,
    recurrence_detection=True,
)
print("EXPECTED vs ACTUAL container/output metadata")
print("field                         expected                         actual")
print(
    "output labels                 3 distinct visible outputs         "
    f"{container_trace.output_layers}"
)
print(
    "duplicate output labels        none                             "
    f"{len(container_trace.output_layers) - len(set(container_trace.output_layers))} duplicate(s)"
)
for label in container_trace.output_layers:
    layer = container_trace.layers[label]
    print(
        f"{label:28s} path/index/name                 "
        f"path={field(layer, 'container_path')} index={field(layer, 'multi_output_index')} "
        f"name={field(layer, 'multi_output_name')} role={field(layer, 'io_role')}"
    )
print(
    "> NOTE: container-path gap: nested dict outputs currently collapse to duplicate "
    "output labels with path=None for the dict members."
)

print("\nEXPECTED vs ACTUAL recurrence/distance metadata")
print("layer          ops  expected distances     actual distances       recurrence evidence")
for layer in container_trace.layers:
    if layer.func_name in {"linear", "relu"}:
        print(
            f"{layer.layer_label:13s} {list(layer.ops)!s:14s} non-None ints          "
            f"input={field(layer, 'min_distance_from_input')} "
            f"output={field(layer, 'min_distance_to_output')} "
            f"equiv={sorted(field(layer, 'equivalent_ops', set()))}"
        )
print(
    "> NOTE: distance-field gap: rolled recurrent layers expose ops/equivalent_ops, "
    "but min_distance_from_input/min_distance_to_output are None in this build."
)

arg_call = next(call for call in container_trace.module_calls if call.address == "arg_block")
fields = [
    "forward_arg_names",
    "num_forward_args_total",
    "num_forward_pos_args",
    "num_forward_kwargs",
    "has_saved_forward_args",
    "forward_args_summary",
    "forward_kwargs_summary",
    "forward_args_template",
    "forward_kwargs_template",
]
print("\nModuleCall argument fields")
for field_name in fields:
    print(f"ModuleCall.{field_name}: {getattr(arg_call, field_name)}")
print(
    "> NOTE: saved-argument-template gap: summaries are retained, but "
    "forward_args_template/forward_kwargs_template are None here."
)
print(
    f"Module vs ModuleCall: {type(container_trace.modules['arg_block']).__name__} / {type(arg_call).__name__}"
)

## 🔧 Sandbox

Mini-experiment: change `target_label`, `show_neighbors`, and `resolve_as_layer`. Expected observation: labels can resolve through either the visible lookup or the layer accessor, and parent/child counts follow the graph position.

In [ ]:
target_label = trace.output_layers[0]
show_neighbors = True
resolve_as_layer = True

target = trace.layers[target_label] if resolve_as_layer else trace[target_label]
print(f"target: {target.layer_label} ({type(target).__name__})")
print(f"module: {target.module}")
print(f"shape: {target.shape}, memory: {target.activation_memory}")
if show_neighbors:
    print(f"parent count: {len(target.parents)} -> {target.parents}")
    print(f"child count: {len(target.children)} -> {target.children}")